# Scenario 3 — Few-Shot In-Context Learning (5 / 10 / 20 examples)
## Decoder Notebook — Qwen2.5-1.5B ICL (NO fine-tuning, NO weight updates)

**Research Question:** Can encoder models (BERT-family) perform in-context learning
comparably to decoder models (Qwen), given the same k examples and zero weight updates?

**Method:** Standard In-Context Learning (prompt-based generation)
- k examples placed directly in the prompt (same format as encoder notebook)
- Model generates 'PHISHING' or 'LEGITIMATE' — weights never updated
- True ICL — identical setup to encoder notebook

**Dataset:** `darkknight25/phishing_benign_email_dataset` — strict 50/50 pool/test split
(Same split as Encoder Notebook — pool: rows 0–99, test: rows 100–199)

**Model:** Qwen2.5-1.5B-Instruct

**K values:** 5, 10, 20  |  **Seeds:** 42, 7, 123, 0, 99

**Comparison pair:** See Encoder Notebook for BERT-family ICL results

In [ ]:
!nvidia-smi
!pip install -q "numpy==1.26.4" "scipy==1.12.0"
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn pandas tqdm

In [1]:
import os, re, time, warnings, random
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────
SEEDS = [42, 7, 123, 0, 99]   # SAME seeds as encoder notebook
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda": print(f"GPU    : {torch.cuda.get_device_name(0)}")

DECODER_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
BNB_CFG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def evaluate(y_true, y_pred, name=""):
    return {
        "Model":     name,
        "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1":        f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}",
    }

print("Setup complete.")

Device : cuda
GPU    : Tesla T4
Setup complete.


In [2]:
# ── Dataset — IDENTICAL split to Encoder Notebook ────────────────
# Pool : rows 0–99   (examples only, never tested)
# Test : rows 100–199 (test only, never used as examples)
print("Loading dataset...")
ds = load_dataset("darkknight25/phishing_benign_email_dataset", split="train")
df = ds.to_pandas()
df["text"]  = (df["subject"].fillna("") + " " + df["body"].fillna("")).str.strip()
df["label"] = (df["label"].astype(str).str.lower() == "phishing").astype(int)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # same shuffle as encoder

pool_df = df.iloc[:100].reset_index(drop=True)
test_df  = df.iloc[100:].reset_index(drop=True)

phish_pool = pool_df[pool_df.label == 1].reset_index(drop=True)
legit_pool  = pool_df[pool_df.label == 0].reset_index(drop=True)

print(f"Pool  : {len(phish_pool)} phishing + {len(legit_pool)} legit (for examples only)")
print(f"Test  : {len(test_df)} emails (held out, never seen as examples)")
print(f"Test label distribution — Phishing: {test_df.label.sum()} | Legit: {(test_df.label==0).sum()}")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

(…)g%20and%20benign%20email%20dataset.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

Pool  : 51 phishing + 49 legit (for examples only)
Test  : 100 emails (held out, never seen as examples)
Test label distribution — Phishing: 49 | Legit: 51


In [3]:
# ── Prompt Builder — IDENTICAL format to Encoder Notebook ─────────
# Both notebooks use the exact same prompt structure
# so any difference in results is due to architecture, not prompt format

def build_icl_prompt(example_texts, example_labels, test_text):
    """
    Build k-shot prompt. Identical format to encoder notebook.
    Examples shuffled randomly per seed.
    """
    lines = ["Classify the following email as PHISHING or LEGITIMATE.\n"]
    combined = list(zip(example_texts, example_labels))
    random.shuffle(combined)   # shuffle order of examples
    for txt, lbl in combined:
        tag = "PHISHING" if lbl == 1 else "LEGITIMATE"
        lines.append(f"Email: {txt[:300]}\nLabel: {tag}\n")
    lines.append(f"Email: {test_text[:400]}\nLabel:")
    return "\n".join(lines)


def run_decoder_icl(model, tokenizer, k, phish_pool, legit_pool, test_df, seed):
    """
    Run Qwen ICL for a given k and random seed.
    Returns list of predictions (0/1).
    No weight updates. No fine-tuning.
    """
    set_seed(seed)

    # Sample k//2 examples from each class — same logic as encoder
    ph_sample = phish_pool.sample(k // 2, random_state=seed)
    lg_sample  = legit_pool.sample(k // 2, random_state=seed)

    example_texts  = list(ph_sample["text"]) + list(lg_sample["text"])
    example_labels = [1] * (k // 2) + [0] * (k // 2)

    preds = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc=f"Qwen {k}-shot seed={seed}", leave=False):
        prompt = build_icl_prompt(example_texts, example_labels, row["text"])
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=2048
        ).to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=5,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False   # greedy — deterministic output
            )

        generated = tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip().upper()

        preds.append(1 if "PHISH" in generated else 0)

    return preds

print("ICL functions defined.")

ICL functions defined.


In [4]:
# ── Main Experiment Loop ──────────────────────────────────────────
# Load Qwen once, run all k values and seeds
# Report mean ± std F1 — matches encoder notebook reporting style

print(f"Loading {DECODER_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(DECODER_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    DECODER_MODEL_ID,
    quantization_config=BNB_CFG,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
# Explicitly confirm no gradients
for param in model.parameters():
    param.requires_grad = False

print("Model loaded. Starting ICL experiment...")

all_results = []
K_VALUES = [5, 10, 20]

for k in K_VALUES:
    print(f"\n--- Qwen {k}-Shot ICL ---")
    seed_f1s, seed_accs = [], []

    for seed in SEEDS:
        preds = run_decoder_icl(
            model, tokenizer, k,
            phish_pool, legit_pool, test_df, seed
        )
        f1  = f1_score(test_df.label, preds, average="binary", zero_division=0)
        acc = accuracy_score(test_df.label, preds)
        seed_f1s.append(f1)
        seed_accs.append(acc)
        print(f"  seed={seed} | F1: {f1:.4f} | Acc: {acc:.4f}")

    mean_f1  = np.mean(seed_f1s)
    std_f1   = np.std(seed_f1s)
    mean_acc = np.mean(seed_accs)

    print(f"  → Mean F1: {mean_f1:.4f} ± {std_f1:.4f} | Mean Acc: {mean_acc:.4f}")
    all_results.append({
        "Architecture": "Decoder",
        "Model":    "Qwen2.5-1.5B ICL",
        "K":        k,
        "Mean F1":  f"{mean_f1:.4f}",
        "Std F1":   f"{std_f1:.4f}",
        "Mean Acc": f"{mean_acc:.4f}",
    })

del model
torch.cuda.empty_cache()

print("\n" + "="*60)
print("SCENARIO 3 — DECODER ICL RESULTS (mean over 5 seeds)")
print("="*60)
results_df = pd.DataFrame(all_results)
print(results_df[["K", "Model", "Mean F1", "Std F1", "Mean Acc"]].to_string(index=False))
print("\nNOTE: No fine-tuning. No weight updates. Pure in-context learning.")
print("Compare these results with the Encoder Notebook (BERT-family ICL).")

Loading Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  seed=123 | F1: 0.8539 | Acc: 0.8700


Qwen 20-shot seed=0:   0%|          | 0/100 [00:00<?, ?it/s]

  seed=0 | F1: 0.8958 | Acc: 0.9000


Qwen 20-shot seed=99:   0%|          | 0/100 [00:00<?, ?it/s]

  seed=99 | F1: 0.9011 | Acc: 0.9100
  → Mean F1: 0.8735 ± 0.0282 | Mean Acc: 0.8860

SCENARIO 3 — DECODER ICL RESULTS (mean over 5 seeds)
 K            Model Mean F1 Std F1 Mean Acc
 5 Qwen2.5-1.5B ICL  0.8382 0.0492   0.8560
10 Qwen2.5-1.5B ICL  0.8496 0.0578   0.8660
20 Qwen2.5-1.5B ICL  0.8735 0.0282   0.8860

NOTE: No fine-tuning. No weight updates. Pure in-context learning.
Compare these results with the Encoder Notebook (BERT-family ICL).
